# Retinal OCT Image Classification with Convolutional Neural Networks

**Objective.** Build a CNN that classifies Optical Coherence Tomography
(OCT) scans of the retina into four diagnostic categories: **CNV**
(choroidal neovascularisation), **DME** (diabetic macular edema),
**DRUSEN**, and **NORMAL**.

**What you will learn:**

| Topic | Section |
|---|---|
| How OCT imaging works | 1 |
| Retinal pathologies visible in OCT | 1 |
| Loading and exploring a medical image dataset | 2 |
| Data augmentation for small medical datasets | 3 |
| Building a CNN classifier in PyTorch | 4 |
| Training with class-weighted loss | 5 |
| Evaluation: confusion matrix, sensitivity, specificity | 6 |
| Grad-CAM: where does the network look? | 7 |

**Dataset:** Kermany et al. (2018), "Identifying Medical Diagnoses and
Treatable Diseases by Image-Based Deep Learning." *Cell* 172(5), 1122-1131.

---
## Setup

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────
# kagglehub is needed to download the dataset
!pip install -q kagglehub

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, datasets
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
from collections import Counter
import random, os, time, glob

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {device}")

---
## 1. Optical Coherence Tomography: Physics and Pathologies

### 1.1 How OCT works

**Optical Coherence Tomography** is an interferometric imaging technique
that produces cross-sectional images of biological tissue with micrometre
resolution.  Key optical principles:

- **Low-coherence interferometry.**  A broadband light source
  ($\lambda \approx 840\;\mathrm{nm}$, bandwidth $\sim 50\;\mathrm{nm}$)
  is split into a reference arm and a sample arm.  Interference occurs
  only when the optical path lengths match within the coherence length
  $l_c = 2\ln 2 / (\pi) \cdot \lambda^2/\Delta\lambda \approx 6\;\mu\mathrm{m}$.

- **Axial resolution** is set by the coherence length (independent of
  focusing optics): $\Delta z \approx l_c$.

- **Lateral resolution** is set by the focusing lens:
  $\Delta x \approx 1.22\,\lambda\,f/D$ (diffraction limit).

- **Spectral-domain OCT (SD-OCT)** uses a spectrometer to capture the
  full interference spectrum in one shot, then applies a Fourier transform
  to recover the depth profile (A-scan).  A 2-D cross-section (B-scan) is
  built by scanning the beam laterally.

### 1.2 Retinal pathologies

| Class | Pathology | OCT appearance |
|-------|-----------|----------------|
| **CNV** | Choroidal neovascularisation | Abnormal blood vessels growing under the retina; fluid accumulation, disrupted outer retinal layers |
| **DME** | Diabetic macular edema | Intraretinal fluid (cystoid spaces), retinal thickening |
| **DRUSEN** | Early AMD marker | Small bright deposits between retinal pigment epithelium (RPE) and Bruch's membrane |
| **NORMAL** | Healthy retina | Clearly defined retinal layers, no fluid or deposits |

---
## 2. Loading the Dataset

We use the Kermany et al. (2018) OCT dataset from Kaggle.  The dataset
is organised as:
```
OCT2017/
  train/
    CNV/    DME/    DRUSEN/    NORMAL/
  test/
    CNV/    DME/    DRUSEN/    NORMAL/
```

Each image is a grayscale retinal B-scan.

In [ ]:
# ── Define transforms ────────────────────────────────────────────────────
IMG_SIZE = 128       # resize to 128x128 for manageable training

# TODO: Define the training and test transforms.
#
# train_transform should include:
#   - Grayscale(num_output_channels=1)
#   - Resize to (IMG_SIZE, IMG_SIZE)
#   - RandomHorizontalFlip()
#   - RandomRotation(10)
#   - RandomAffine(degrees=0, translate=(0.05, 0.05))
#   - ToTensor()
#   - Normalize(mean=[0.5], std=[0.5])
#
# test_transform should include:
#   - Grayscale(num_output_channels=1)
#   - Resize to (IMG_SIZE, IMG_SIZE)
#   - ToTensor()
#   - Normalize(mean=[0.5], std=[0.5])
#
# Hint: Use transforms.Compose([...])

train_transform = ...  # YOUR CODE HERE
raise NotImplementedError("Define train_transform")

test_transform = ...   # YOUR CODE HERE
raise NotImplementedError("Define test_transform")

train_full = datasets.ImageFolder(os.path.join(DATA_ROOT, "train"),
                                  transform=train_transform)
test_ds    = datasets.ImageFolder(os.path.join(DATA_ROOT, "test"),
                                  transform=test_transform)

CLASS_NAMES = train_full.classes
NUM_CLASSES = len(CLASS_NAMES)
print(f"Classes: {CLASS_NAMES}")
print(f"Training images: {len(train_full)}")
print(f"Test images:     {len(test_ds)}")

# Class distribution
train_counts = Counter(train_full.targets)
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name}: {train_counts[i]}")

In [ ]:
# ── Create a validation split (10% of training data) ────────────────────
from torch.utils.data import random_split

n_total = len(train_full)
n_val = int(0.1 * n_total)
n_train = n_total - n_val

train_ds, val_ds_raw = random_split(
    train_full, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED)
)

# DataLoaders
BATCH = 32
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                       num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds_raw, batch_size=BATCH, num_workers=2,
                       pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH, num_workers=2,
                       pin_memory=True)

print(f"train {n_train} | val {n_val} | test {len(test_ds)}")
xb, yb = next(iter(train_dl))
print(f"Batch: images {xb.shape}, labels {yb.shape}")

### 2.1 Visualise sample images

---
## 3. Data Augmentation and Class Imbalance

Medical imaging datasets are often small and imbalanced.  **Data
augmentation** artificially increases the training set by applying random
transformations that preserve the diagnostic content.

### 3.1 Class imbalance

The dataset is imbalanced.  We handle this with a **weighted loss
function** that penalises mistakes on rare classes more heavily:

$$
w_c = \frac{N_{\text{total}}}{C \cdot N_c}
$$

where $N_{\text{total}}$ is the total number of samples, $C$ is the
number of classes, and $N_c$ is the number of samples in class $c$.

In [ ]:
# ── Compute class weights ────────────────────────────────────────────────
# TODO: Compute class weights to handle imbalanced data.
#
# Steps:
#   1. Get targets as numpy array: targets = np.array(train_full.targets)
#   2. Count samples per class: class_counts = np.bincount(targets, minlength=NUM_CLASSES)
#   3. Compute weights: w_c = N_total / (C * N_c)
#   4. Convert to a torch tensor on the device
#
# Expected output: a tensor of shape (4,) with one weight per class

targets = np.array(train_full.targets)
class_counts = np.bincount(targets, minlength=NUM_CLASSES)

class_weights = ...  # YOUR CODE HERE
raise NotImplementedError("Compute class_weights")

class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

print("Class weights:")
for name, w, n in zip(CLASS_NAMES, class_weights, class_counts):
    print(f"  {name:8s}: weight={w:.3f}  (n={n})")

---
## 4. Building the network


### Why a CNN for images?

Unlike 1-D data (e.g., spectra), OCT images are 2-D and the pathological
features can appear at different locations.  **Convolutional layers** provide:

- **Translation invariance**: detect a feature regardless of position
- **Parameter sharing**: the same filter scans the entire image
- **Hierarchical features**: early layers detect edges, later layers
  detect complex structures (fluid pockets, drusen deposits)

### Architecture

Build a CNN with 4 convolutional blocks followed by global average pooling
and a linear classifier:

```
Input (1, 128, 128)
  -> Conv(1, 32, 3) + BN + ReLU + MaxPool(2)    -> (32, 64, 64)
  -> Conv(32, 64, 3) + BN + ReLU + MaxPool(2)   -> (64, 32, 32)
  -> Conv(64, 128, 3) + BN + ReLU + MaxPool(2)  -> (128, 16, 16)
  -> Conv(128, 256, 3) + BN + ReLU + MaxPool(2) -> (256, 8, 8)
  -> AdaptiveAvgPool(1, 1)                       -> (256,)
  -> Dropout(0.5) -> Linear(256, 4)
```

In [ ]:
# ── CNN definition ───────────────────────────────────────────────────────
# TODO: Complete the OCTCNN class.
#
# The network should have:
#   self.features: nn.Sequential with 4 blocks, each containing
#       nn.Conv2d(in, out, 3, padding=1), nn.BatchNorm2d(out),
#       nn.ReLU(), nn.MaxPool2d(2)
#     Channel progression: 1 -> 32 -> 64 -> 128 -> 256
#   self.pool: nn.AdaptiveAvgPool2d((1, 1))
#   self.classifier: nn.Sequential(nn.Dropout(0.5), nn.Linear(256, n_classes))

class OCTCNN(nn.Module):
    def __init__(self, n_classes=4):
        super().__init__()

        # TODO: Define self.features with 4 conv blocks
        self.features = ...  # YOUR CODE HERE
        raise NotImplementedError("Define self.features")

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

model = OCTCNN(NUM_CLASSES).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")

# Verify output shape
dummy = torch.randn(2, 1, IMG_SIZE, IMG_SIZE).to(device)
print(f"Output shape: {model(dummy).shape}")

---
## 5. Training

- **Loss**: cross-entropy with class weights (Section 3.1)
- **Optimiser**: Adam, learning rate $10^{-3}$
- **Scheduler**: cosine annealing over the training run
- **Epochs**: 15 (sufficient with augmentation on GPU)

In [ ]:
# ── Training setup ───────────────────────────────────────────────────────
EPOCHS = 15
LR = 1e-3

# TODO: Define loss function, optimiser, and scheduler.
#
# criterion: nn.CrossEntropyLoss with the class_weights computed above
# optimiser: torch.optim.Adam with lr=LR
# scheduler: torch.optim.lr_scheduler.CosineAnnealingLR with T_max=EPOCHS

criterion = ...  # YOUR CODE HERE
optimiser = ...  # YOUR CODE HERE
scheduler = ...  # YOUR CODE HERE
raise NotImplementedError("Define criterion, optimiser, and scheduler")

In [ ]:
# ── Training loop ────────────────────────────────────────────────────────
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

def run_epoch(loader, train=True):
    model.train(train)
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            if train:
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
            total_loss += loss.item() * len(yb)
            correct += (logits.argmax(1) == yb).sum().item()
            total += len(yb)
    return total_loss / total, correct / total

best_val_acc = 0.0
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tl, ta = run_epoch(train_dl, train=True)
    vl, va = run_epoch(val_dl, train=False)
    scheduler.step()
    history["train_loss"].append(tl)
    history["val_loss"].append(vl)
    history["train_acc"].append(ta)
    history["val_acc"].append(va)
    if va > best_val_acc:
        best_val_acc = va
        torch.save(model.state_dict(), "best_oct_model.pth")
    lr_now = optimiser.param_groups[0]["lr"]
    print(f"epoch {epoch:2d} | train {tl:.4f}/{ta:.3f} | "
          f"val {vl:.4f}/{va:.3f} | lr {lr_now:.1e} | {time.time()-t0:.0f}s")

print(f"\nBest val accuracy: {best_val_acc:.3f}")
model.load_state_dict(torch.load("best_oct_model.pth", weights_only=True))

In [ ]:
# ── Training curves ──────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history["train_loss"], label="train"); ax1.plot(history["val_loss"], label="val")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.legend(); ax1.set_title("Loss")

ax2.plot(history["train_acc"],  label="train"); ax2.plot(history["val_acc"],  label="val")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy"); ax2.legend(); ax2.set_title("Accuracy")

plt.tight_layout(); plt.show()

---
## 6. Evaluation

For medical imaging, **sensitivity** (recall) and **specificity** are
especially important:

- **Sensitivity** = TP / (TP + FN) -- ability to detect disease (a
  missed diagnosis is dangerous)
- **Specificity** = TN / (TN + FP) -- ability to confirm health (a
  false alarm causes unnecessary anxiety and cost)

In [ ]:
# ── Test evaluation ──────────────────────────────────────────────────────
# TODO: Evaluate the model on the test set.
#
# Steps:
#   1. Set model to eval mode
#   2. Loop over test_dl with torch.no_grad()
#   3. Collect predictions and labels
#   4. Compute and print accuracy
#   5. Print classification_report

model.eval()
all_preds, all_labels = [], []

# YOUR CODE HERE: loop over test_dl, collect predictions
raise NotImplementedError("Evaluate the model on the test set")

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()
test_acc = (all_preds == all_labels).mean()
print(f"TEST accuracy: {test_acc:.4f}  ({(all_preds != all_labels).sum()} "
      f"mistakes out of {len(all_labels)})\n")

print(classification_report(all_labels, all_preds,
                            target_names=CLASS_NAMES, digits=3))

In [ ]:
# ── Confusion matrix ─────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES)
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion Matrix")

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        val = cm[i, j]
        colour = "white" if val > cm.max() / 2 else "black"
        ax.text(j, i, str(val), ha="center", va="center",
                color=colour, fontsize=10)

plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

---
## 7. Grad-CAM: Where Does the Network Look?

**Gradient-weighted Class Activation Mapping** (Selvaraju et al., 2017)
highlights which regions of the image are most important for the predicted
class.  For a target class $c$ and the feature maps $A^k$ of the last
convolutional layer:

$$
\alpha_k^c = \frac{1}{Z} \sum_i \sum_j
\frac{\partial y^c}{\partial A^k_{ij}}
$$

$$
L_{\text{Grad-CAM}}^c = \mathrm{ReLU}\!\left(\sum_k \alpha_k^c\, A^k\right)
$$

In [ ]:
# ── Grad-CAM implementation ──────────────────────────────────────────────
# TODO: Complete the GradCAM class.
#
# The class needs:
#   __init__: register a forward hook on the target layer
#   _hook: save the feature map output; call out.requires_grad_(True);
#          register a backward hook to save gradients
#   __call__: forward pass, backward on target class score,
#             compute weighted combination, upsample to input size
#
# Hints:
#   - Use layer.register_forward_hook(self._hook)
#   - In the hook: out.register_hook(lambda g: setattr(self, "grads", g))
#   - alpha = self.grads.mean(dim=(2, 3), keepdim=True)
#   - cam = F.relu((alpha * self.acts).sum(1, keepdim=True))
#   - Use F.interpolate to upsample cam to input size

class GradCAM:
    def __init__(self, model, layer):
        self.model, self.acts, self.grads = model, None, None
        # YOUR CODE HERE: register forward hook
        raise NotImplementedError("Register forward hook")

    def _hook(self, module, inp, out):
        # YOUR CODE HERE: save activations, enable grad, register backward hook
        raise NotImplementedError("Implement _hook")

    def __call__(self, x, target=None):
        self.model.eval()
        logits = self.model(x)
        c = logits.argmax(1).item() if target is None else target
        self.model.zero_grad()
        logits[0, c].backward()
        # YOUR CODE HERE: compute and return the CAM
        raise NotImplementedError("Compute Grad-CAM")

    def remove(self):
        self.handle.remove()

In [ ]:
# ── Visualise Grad-CAM ──────────────────────────────────────────────────
# Hook into the last conv layer
last_conv = model.features[-4]   # last Conv2d in block 4
gradcam = GradCAM(model, last_conv)

fig, axes = plt.subplots(4, 4, figsize=(14, 14))

for row in range(4):
    cls_indices = [i for i, t in enumerate(test_ds.targets) if t == row]
    idx = cls_indices[0]
    img, label = test_ds[idx]
    x = img.unsqueeze(0).to(device)

    cam, pred = gradcam(x)
    img_np = img[0].numpy() * 0.5 + 0.5

    # Original
    axes[row, 0].imshow(img_np, cmap="gray"); axes[row, 0].axis("off")
    axes[row, 0].set_ylabel(CLASS_NAMES[label], fontsize=12,
                           fontweight="bold", rotation=0, labelpad=50,
                           va="center")
    if row == 0: axes[row, 0].set_title("Original")

    # Grad-CAM heatmap
    axes[row, 1].imshow(cam.numpy(), cmap="jet"); axes[row, 1].axis("off")
    if row == 0: axes[row, 1].set_title("Grad-CAM")

    # Overlay
    axes[row, 2].imshow(img_np, cmap="gray")
    axes[row, 2].imshow(cam.numpy(), cmap="jet", alpha=0.45)
    axes[row, 2].axis("off")
    pred_name = CLASS_NAMES[pred]
    axes[row, 2].set_title(f"pred: {pred_name}" if row == 0 else pred_name,
                          fontsize=10)
    if row == 0: axes[row, 2].set_title("Overlay")

    # Another example
    if len(cls_indices) > 1:
        idx2 = cls_indices[1]
        img2, _ = test_ds[idx2]
        x2 = img2.unsqueeze(0).to(device)
        cam2, pred2 = gradcam(x2)
        img2_np = img2[0].numpy() * 0.5 + 0.5
        axes[row, 3].imshow(img2_np, cmap="gray")
        axes[row, 3].imshow(cam2.numpy(), cmap="jet", alpha=0.45)
    axes[row, 3].axis("off")
    if row == 0: axes[row, 3].set_title("Another example")

plt.suptitle("Grad-CAM visualisation by class", fontsize=14, y=1.01)
plt.tight_layout(); plt.show()
gradcam.remove()

### Interpreting Grad-CAM

Examine the heatmaps:

- **CNV**: does the network focus on the sub-retinal fluid and
  neovascular membrane region?
- **DME**: does it highlight the intraretinal cystoid spaces?
- **DRUSEN**: does it attend to the RPE-Bruch's membrane interface
  where drusen deposits sit?
- **NORMAL**: attention should be diffuse (no localised pathology).

> **Clinical relevance:** If the model focuses on artefacts (e.g., image
> borders, text overlays) rather than retinal structures, it may not
> generalise to scans from different machines or hospitals.

---
## 8. Further Exercises

**Exercise 1 -- Transfer learning.**
Replace the custom CNN with a pretrained ResNet-18.  Modify the first
conv layer to accept 1-channel input and the final layer for 4 classes.
Compare accuracy and training speed.

**Exercise 2 -- Image resolution.**
Retrain with `IMG_SIZE = 224` instead of 128.  Does higher resolution
improve accuracy?  How does training time change?

**Exercise 3 -- Error analysis.**
Find the most confidently wrong predictions (highest softmax probability
for the wrong class).  Display them with Grad-CAM.  Are these ambiguous
cases or model failures?

**Exercise 4 -- Cross-device generalisation.**
The Kermany dataset comes from a single OCT device.  Discuss what might
happen if you applied this model to scans from a different manufacturer.
What domain adaptation strategies could help?

In [ ]:
# ── Starter: Exercise 1 (Transfer learning) ─────────────────────────────
from torchvision import models

# Load pretrained ResNet-18
resnet = models.resnet18(weights="IMAGENET1K_V1")

# TODO:
# 1. Replace resnet.conv1 to accept 1 input channel instead of 3
#    Hint: nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
# 2. Replace resnet.fc to output 4 classes instead of 1000
#    Hint: nn.Linear(resnet.fc.in_features, NUM_CLASSES)
# 3. Move to device, train, and compare with the custom CNN

---
*Notebook prepared for optics students learning medical image
classification.  The model is for educational purposes only and must not
be used for clinical diagnosis.*